<a href="https://colab.research.google.com/github/dbaglodi/CREPE-YOLO/blob/main/CREPE_YOLO_eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---
## 0 · Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

---
## 1 · Install Dependencies

In [ ]:
# Core audio / ML / evaluation packages
!pip install -q crepe mir_eval librosa soundfile scipy matplotlib tqdm

# note_seq: MIDI I/O library used by MT3
!pip install -q note_seq==0.0.5

import crepe, mir_eval, librosa, soundfile, note_seq
print('All dependencies installed')
print(f'  crepe    {crepe.__version__}')
print(f'  mir_eval {mir_eval.__version__}')
print(f'  librosa  {librosa.__version__}')
print(f'  note_seq {note_seq.__version__}')

---
## 2 · Paths & Global Constants

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess
import numpy as np

# ── Dataset (Google Drive layout) ────────────────────────────────────────────
#   GT-ITM-Flute-99/
#     audio/          <- .wav files
#     annotations/    <- .txt files
DATASET_ROOT = Path('/content/drive/MyDrive/DL Project/GT-ITM-Flute-99')
AUDIO_DIR    = DATASET_ROOT / 'audio'
ANNOT_DIR    = DATASET_ROOT / 'annotations'

# ── Colab working directories (local SSD) ────────────────────────────────────
WORK_DIR      = Path('/content/crepe_yolo')
PROCESSED_DIR = WORK_DIR / 'processed' / 'itm_flute'
SPLIT_DIR     = WORK_DIR / 'configs'
PRED_DIR      = WORK_DIR / 'predictions'
EVAL_DIR      = WORK_DIR / 'evaluation'

for d in [PROCESSED_DIR, SPLIT_DIR,
          PRED_DIR / 'pyin',
          PRED_DIR / 'crepe_notes',
          PRED_DIR / 'mt3',
          EVAL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Audio preprocessing target ───────────────────────────────────────────────
RESAMPLE_SR = 16000   # 16 kHz mono — CREPE's expected input

# ── mir_eval tolerances (matching CREPE Notes / MT3 papers) ──────────────────
ONSET_TOL    = 0.05   # 50 ms
OFFSET_RATIO = 0.20   # 20% of note duration
OFFSET_MIN   = 0.05   # 50 ms minimum
PITCH_TOL    = 0.5    # +/- 0.5 semitones

assert DATASET_ROOT.exists(), f'Dataset root not found: {DATASET_ROOT}'
assert AUDIO_DIR.exists(),    f'audio/ subfolder not found: {AUDIO_DIR}'
assert ANNOT_DIR.exists(),    f'annotations/ subfolder not found: {ANNOT_DIR}'

AUDIO_FILES = sorted(AUDIO_DIR.glob('*.wav'))
print(f'Dataset root   : {DATASET_ROOT}')
print(f'Audio files    : {len(AUDIO_FILES)}')
print(f'Annotation dir : {ANNOT_DIR}')
print(f'Working dir    : {WORK_DIR}')


---
## 3 · Inspect Dataset & Parse Annotations

In [ ]:
import soundfile as sf
from tqdm.notebook import tqdm

# Print a sample annotation to confirm the column format
sample_txts = sorted(ANNOT_DIR.glob('*.txt'))
if sample_txts:
    print(f'Sample annotation: {sample_txts[0].name}')
    print('Columns (tab-separated):')
    print('  [0]index  [1]source_file  [2]onset  [3]offset  [4]duration  [5]type  [6]note_name  [7]freq_hz')
    print('-' * 90)
    with open(sample_txts[0]) as fh:
        for idx, line in enumerate(fh):
            print(f'  {idx:3d}: {line.rstrip()}')
            if idx >= 9: print('  ...'); break
    print()

# Show the naming mismatch so it is clear what the prefix-stripping fixes
annot_example = sample_txts[0].stem if sample_txts else 'none'
audio_example = AUDIO_FILES[0].stem if AUDIO_FILES else 'none'
print(f'Audio stem example : {audio_example}')
print(f'Annot stem example : {annot_example}')
print('  => the izzy_GT_ prefix will be stripped so stems match')
print()

# ── Annotation parser ─────────────────────────────────────────────────────────
# Tab-separated 8-column format:
#   col 0: row index      col 1: source filename   col 2: onset (s)
#   col 3: offset (s)    col 4: duration (s)       col 5: note type (NOTE/ct/...)
#   col 6: note name (e.g. C#5, Bb4)               col 7: frequency (Hz)
#
# Annotation files are named  izzy_GT_<audio_stem>.txt
# Audio files are named        <audio_stem>.wav   (no prefix)
NOTE_MAP = {'C': 0, 'D': 2, 'E': 4, 'F': 5, 'G': 7, 'A': 9, 'B': 11}

def note_name_to_midi(name):
    name = name.strip()
    i = len(name) - 1
    while i >= 0 and (name[i].isdigit() or name[i] == '-'):
        i -= 1
    note_part  = name[:i+1]
    octave_str = name[i+1:]
    octave     = int(octave_str) if octave_str else 4
    semitone   = NOTE_MAP[note_part[0].upper()]
    if '#' in note_part: semitone += 1
    elif 'b' in note_part: semitone -= 1
    return 12 * (octave + 1) + semitone

def parse_annotation(path):
    """Parse ITM-Flute-99 tab-separated annotation file.
    Returns list of {onset, offset, pitch_midi}."""
    notes = []
    with open(path) as fh:
        for line in fh:
            line = line.strip()
            if not line or line.startswith('#'): continue
            parts = line.split('\t')         # tab-separated
            if len(parts) < 7: continue
            try:
                onset  = float(parts[2])     # col 2: onset time
                offset = float(parts[3])     # col 3: offset time
                p_raw  = parts[6].strip()    # col 6: note name
            except (ValueError, IndexError):
                continue
            try:
                pitch_midi = note_name_to_midi(p_raw)
            except Exception:
                continue  # skip rows with unparseable pitch (e.g. silence markers)
            notes.append({'onset': onset, 'offset': offset, 'pitch_midi': pitch_midi})
    return notes

# Build a lookup: audio_stem -> annotation Path
# Strip the 'izzy_GT_' prefix from annotation filenames to get the matching audio stem
_all_annot   = list(ANNOT_DIR.glob('*.txt'))
_annot_lookup = {}
for p in _all_annot:
    key = p.stem
    for prefix in ['izzy_GT_', 'GT_', 'izzy_']:
        if key.startswith(prefix):
            key = key[len(prefix):]
            break
    _annot_lookup[key] = p

def find_annotation(stem):
    if stem in _annot_lookup:
        return _annot_lookup[stem]
    # Fallback: substring search for any unexpected prefix variant
    for p in _all_annot:
        if p.stem.endswith(stem):
            return p
    return None

print(f'Annotation files found : {len(_all_annot)}')
print(f'Lookup table size      : {len(_annot_lookup)}')
if _annot_lookup:
    k0 = next(iter(_annot_lookup))
    print(f'Example: "{k0}" -> {_annot_lookup[k0].name}')

# ── Build dataset_info list ───────────────────────────────────────────────────
dataset_info, missing = [], []
for wav in tqdm(AUDIO_FILES, desc='Scanning'):
    annot = find_annotation(wav.stem)
    if annot is None:
        missing.append(wav.stem); continue
    try:
        info  = sf.info(str(wav))
        notes = parse_annotation(annot)
        dataset_info.append({
            'stem':       wav.stem,
            'wav_path':   wav,
            'annot_path': annot,
            'duration':   info.duration,
            'sr':         info.samplerate,
            'n_notes':    len(notes),
        })
    except Exception as e:
        print(f'  [WARN] {wav.stem}: {e}')

total_dur   = sum(d['duration'] for d in dataset_info)
total_notes = sum(d['n_notes']  for d in dataset_info)
print(f'\n{"="*50}')
print( '  ITM-Flute-99 Dataset Summary')
print(f'{"="*50}')
print(f'  Pieces with audio+annotation : {len(dataset_info)}')
print(f'  Missing annotations          : {len(missing)}')
if missing: print(f'    -> {missing[:5]}')
if dataset_info:
    print(f'  Total audio duration         : {total_dur/60:.1f} min')
    print(f'  Total annotated notes        : {total_notes}')
    print(f'  Avg notes / piece            : {total_notes/len(dataset_info):.0f}')
    print(f'  Sample rates found           : {set(d["sr"] for d in dataset_info)}')
else:
    print('  No pieces loaded - check annotation prefix logic above.')


---
## 4 · Generate Train/Test Splits


In [ ]:
all_stems = [d['stem'] for d in dataset_info]
n         = len(all_stems)

folds = [
    {'fold': i, 'test': [all_stems[i]], 'train': all_stems[:i] + all_stems[i+1:]}
    for i in range(n)
]
split_data = {
    'dataset':    'ITM-Flute-99',
    'reference':  'Koekuer et al. (2019) leave-one-out CV',
    'all_pieces': all_stems,
    'n_pieces':   n,
    'folds':      folds,
}
split_path = SPLIT_DIR / 'itm_flute_splits.json'
with open(split_path, 'w') as f:
    json.dump(split_data, f, indent=2)

print(f'Generated {n} leave-one-out folds  ->  {split_path}')
print(f'Example fold 0:')
print(f'  test  = {folds[0]["test"]}')
print(f'  train = {folds[0]["train"][:3]} ... ({len(folds[0]["train"])} pieces)')

---
## 5 · Preprocess Audio & Clean Annotations


In [ ]:
import soundfile as sf

def midi_to_hz(m):    return 440.0 * (2 ** ((m - 69) / 12))

def clean_notes(raw, duration):
    """Remove malformed notes, sort by onset, clamp to audio length."""
    out = []
    for n in raw:
        on  = float(n.get('onset',  n.get('onset_time',  0)))
        off = float(n.get('offset', n.get('offset_time', 0)))
        pm  = n.get('pitch_midi', n.get('pitch'))
        if pm is None: continue
        pm = int(pm)
        if not (21 <= pm <= 108): continue   # outside piano/flute range
        if off <= on: continue               # zero or negative duration
        off = min(off, duration)             # clamp to audio length
        if off <= on: continue
        out.append({'onset': on, 'offset': off, 'pitch_midi': pm})
    return sorted(out, key=lambda x: x['onset'])

def resample_normalize(src, dst):
    """Resample to 16 kHz mono + EBU R128 loudness normalization."""
    r = subprocess.run([
        'ffmpeg', '-y', '-i', str(src),
        '-ac', '1', '-ar', str(RESAMPLE_SR),
        '-filter:a', 'loudnorm=I=-23:LRA=11:TP=-2',
        str(dst), '-loglevel', 'error'
    ], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f'ffmpeg: {r.stderr[:200]}')

skipped = []
for entry in tqdm(dataset_info, desc='Preprocessing'):
    stem    = entry['stem']
    out_dir = PROCESSED_DIR / stem
    out_dir.mkdir(parents=True, exist_ok=True)
    wav_dst = out_dir / 'audio_16k.wav'
    try:
        if not wav_dst.exists():
            resample_normalize(entry['wav_path'], wav_dst)
        duration = sf.info(str(wav_dst)).duration
        raw      = parse_annotation(entry['annot_path'])
        notes    = clean_notes(raw, duration)
        with open(out_dir / 'notes.json', 'w') as f:
            json.dump(notes, f, indent=2)
    except Exception as e:
        print(f'  [ERROR] {stem}: {e}'); skipped.append(stem)

processed = [d['stem'] for d in dataset_info if d['stem'] not in skipped]
print(f'Preprocessed : {len(processed)} pieces  |  Skipped: {len(skipped)}')

# Quick sanity check
s0 = processed[0]
with open(PROCESSED_DIR / s0 / 'notes.json') as f: nn = json.load(f)
print(f'\nSample "{s0}": {len(nn)} cleaned notes')
print(f'  First note: onset={nn[0]["onset"]:.3f}s  offset={nn[0]["offset"]:.3f}s  pitch_midi={nn[0]["pitch_midi"]}')


---
## 6 · Piano Roll Visualisation

In [ ]:
import matplotlib.pyplot as plt

def plot_piano_roll(notes, title='', max_sec=30.0):
    fig, ax = plt.subplots(figsize=(14, 4))
    for n in notes:
        if n['onset'] >= max_sec: continue
        dur = min(n['offset'], max_sec) - n['onset']
        ax.barh(n['pitch_midi'], dur, left=n['onset'], height=0.8,
                color='steelblue', alpha=0.75, edgecolor='white', linewidth=0.3)
    ax.set_xlabel('Time (s)', fontsize=11)
    ax.set_ylabel('MIDI pitch', fontsize=11)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlim(0, max_sec)
    ax.grid(axis='x', alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()
    plt.show()

with open(PROCESSED_DIR / processed[0] / 'notes.json') as f:
    sample_notes = json.load(f)
plot_piano_roll(sample_notes, title=f'Piano Roll  {processed[0]}  (first 30 s)')
print(f'{len([n for n in sample_notes if n["onset"] < 30])} notes visible')

---
## 7 · Baseline 1 — pYIN

In [ ]:
import librosa

def voiced_frames_to_notes(f0, voiced, hop_s, min_note_s=0.05):
    notes, in_note, note_start, pitches = [], False, 0, []
    def flush(end):
        if (end - note_start) * hop_s >= min_note_s:
            notes.append({'onset':      note_start * hop_s,
                          'offset':     end * hop_s,
                          'pitch_midi': int(round(np.mean(pitches)))})
    for i, (freq, v) in enumerate(zip(f0, voiced)):
        if v and freq is not None and not np.isnan(freq) and freq > 0:
            midi = 69 + 12 * np.log2(freq / 440.0)
            if not in_note:
                in_note, note_start, pitches = True, i, [midi]
            elif abs(midi - np.mean(pitches)) < 1.0:
                pitches.append(midi)
            else:
                flush(i); note_start, pitches = i, [midi]
        elif in_note:
            flush(i); in_note = False; pitches = []
    if in_note: flush(len(f0))
    return notes

def run_pyin(wav_path):
    y, sr = librosa.load(str(wav_path), sr=RESAMPLE_SR, mono=True)
    hop   = 160   # 10 ms at 16 kHz
    f0, voiced, _ = librosa.pyin(
        y, fmin=librosa.note_to_hz('C2'), fmax=librosa.note_to_hz('C7'),
        sr=sr, hop_length=hop, fill_na=None,
    )
    return voiced_frames_to_notes(f0, voiced, hop / sr)

pyin_dir = PRED_DIR / 'pyin'
for entry in tqdm(dataset_info, desc='pYIN'):
    out = pyin_dir / f'{entry["stem"]}.json'
    if out.exists(): continue
    wav = PROCESSED_DIR / entry['stem'] / 'audio_16k.wav'
    if not wav.exists(): continue
    try:
        notes = run_pyin(wav)
        with open(out, 'w') as f: json.dump(notes, f, indent=2)
    except Exception as e:
        print(f'  [ERROR] {entry["stem"]}: {e}')

print(f'pYIN complete: {len(list(pyin_dir.glob("*.json")))}/{len(dataset_info)} predictions')

---
## 8 · Baseline 2 — CREPE Notes

In [ ]:
import crepe
from scipy.ndimage import median_filter

def run_crepe_notes(wav_path, conf_threshold=0.5):
    audio, sr = sf.read(str(wav_path))
    if audio.ndim > 1: audio = audio.mean(axis=1)
    _, frequency, confidence, _ = crepe.predict(
        audio, sr,
        model_capacity='small',   # use 'full' for best accuracy
        viterbi=True,
        step_size=10,
        verbose=0,
    )
    voiced  = confidence >= conf_threshold
    f0_smth = median_filter(np.where(voiced, frequency, 0.0), size=5)
    return voiced_frames_to_notes(np.where(voiced, f0_smth, np.nan), voiced, hop_s=0.010)

crepe_dir = PRED_DIR / 'crepe_notes'
for entry in tqdm(dataset_info, desc='CREPE Notes'):
    out = crepe_dir / f'{entry["stem"]}.json'
    if out.exists(): continue
    wav = PROCESSED_DIR / entry['stem'] / 'audio_16k.wav'
    if not wav.exists(): continue
    try:
        notes = run_crepe_notes(wav)
        with open(out, 'w') as f: json.dump(notes, f, indent=2)
    except Exception as e:
        print(f'  [ERROR] {entry["stem"]}: {e}')

print(f'CREPE Notes complete: {len(list(crepe_dir.glob("*.json")))}/{len(dataset_info)} predictions')

---
## 9 · Baseline 3 — MT3


In [ ]:
# ── 9a: Clone MT3 repo and install using the official Colab commands ──
# Source: magenta/mt3/blob/main/mt3/colab/music_transcription_with_transformers.ipynb

import os
from pathlib import Path

MT3_REPO = Path('/content/mt3')

if not MT3_REPO.exists():
    !git clone --quiet https://github.com/magenta/mt3.git /content/mt3
    print('Cloned MT3 repo')
else:
    print('MT3 repo already present')

%cd /content/mt3

# Exact install command from the official MT3 Colab notebook:
!apt-get update -qq && apt-get install -qq libfluidsynth3 build-essential libasound2-dev libjack-dev
!python3 -m pip install jax[cuda12] nest-asyncio pyfluidsynth==1.3.0 -e . -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html

# [FIX] Explicitly reinstall/upgrade tensorflow and tensorflow-text to resolve potential symbol conflicts
# This ensures that tensorflow and tensorflow-text are installed in a compatible state.
!pip install --upgrade tensorflow tensorflow-text

%cd /content

print()
print('=' * 55)
print('IMPORTANT: You must restart the kernel now.')
print('Go to Runtime -> Restart session, then:')
print('  1. Re-run cells 0-8 to restore dataset_info')
print('  2. Continue from cell 9b')
print('=' * 55)


In [ ]:
# ── 9b: Download official MT3 checkpoint ─────────────────────────────────────
# Run this after restarting the kernel and re-running cells 0-8.
from pathlib import Path
import subprocess

MT3_CKPT = Path('/content/checkpoints')

if not MT3_CKPT.exists():
    print('Downloading MT3 checkpoints (~1.5 GB) ...')
    # Official gsutil command from the MT3 Colab notebook
    !gsutil -q -m cp -r gs://mt3/checkpoints /content/
    print('Checkpoints downloaded.')
else:
    print(f'Checkpoints already present at {MT3_CKPT}')

print('Available checkpoints:')
for p in sorted(MT3_CKPT.iterdir()):
    print(f'  {p.name}')
    # We want the mt3 checkpoint (multi-instrument), not ismir2021 (piano only)


In [ ]:
# ── 9c: MT3 inference helper (Self-contained InferenceModel) ──
import sys
import os
import traceback

if '/content/mt3' not in sys.path:
    sys.path.insert(0, '/content/mt3')

import jax
import numpy as np
import tensorflow as tf
from mt3 import models
from mt3 import network
from mt3 import preprocessors
from mt3 import vocabularies
from mt3 import spectrograms
import note_seq
import t5x

def get_mt3_vocab():
    """Recreate the default MT3 vocabulary with required config."""
    config = vocabularies.VocabularyConfig()
    codec = vocabularies.build_codec(vocab_config=config)
    return vocabularies.vocabulary_from_codec(codec)

class InferenceModel(object):
  def __init__(self, checkpoint_path, model_type='mt3'):
    checkpoint_path = os.path.expanduser(checkpoint_path)

    self.vocab = get_mt3_vocab()
    self.spectrogram_config = spectrograms.SpectrogramConfig()

    # UPDATED: vocab_size=1536 to match the checkpoint weights
    config = network.T5Config(
        vocab_size=1536,          # Matched to checkpoint shape (1536, 512)
        emb_dim=512,
        num_heads=8,
        num_encoder_layers=8,
        num_decoder_layers=8,
        head_dim=48,
        mlp_dim=1024,
        mlp_activations=('gelu', 'linear'),
        dropout_rate=0.0,
        logits_via_embedding=False)

    # Initialize hyper-parameters with required defaults for inference-only usage
    hp = t5x.adafactor._AdafactorHyperParams(
        learning_rate=0.0,
        factored=True,
        multiply_by_parameter_scale=True,
        beta1=None,
        decay_rate=0.8,
        step_offset=0,
        clipping_threshold=1.0,
        weight_decay_rate=None,
        min_dim_size_to_factor=128,
        epsilon1=1e-30,
        epsilon2=1e-3)

    optimizer_def = t5x.adafactor.OptimizerDef(hyper_params=hp)

    # input_depth=512 is the standard for MT3 mel-spectrogram features
    self.model = models.ContinuousInputsEncoderDecoderModel(
        module=network.Transformer(config),
        input_vocabulary=vocabularies.GenericTokenVocabulary(regular_ids=range(0)),
        output_vocabulary=self.vocab,
        optimizer_def=optimizer_def,
        input_depth=512)

    self.restore_checkpoint(checkpoint_path)

  def restore_checkpoint(self, checkpoint_path):
    ckpt_data = t5x.checkpoints.load_t5x_checkpoint(checkpoint_path)
    if 'target' in ckpt_data:
        self.variables = ckpt_data['target']
    else:
        self.variables = ckpt_data

  def __call__(self, audio):
    # Compute spectrogram directly
    spec = spectrograms.compute_spectrogram(audio.astype(np.float32), self.spectrogram_config)

    # Reshape to [batch, length, depth]
    if spec.ndim == 2:
        spec = spec[np.newaxis, ...]

    # Wrap in the dictionary format expected by T5X
    batch = {
        'inputs': spec,
        'encoder_input_tokens': np.zeros((spec.shape[0], spec.shape[1], 512), dtype=np.int32),
        'decoder_input_tokens': np.zeros((spec.shape[0], 1), dtype=np.int32)
    }

    prediction = self.model.predict_batch_with_aux(self.variables, batch)
    tokens = prediction[0] if isinstance(prediction, (tuple, list)) else prediction

    # [FIX] Remove batch dimension before decoding, ensuring it's at least 1D
    if tokens.ndim > 1:
        tokens = np.squeeze(tokens)
    if tokens.ndim == 0:
        tokens = np.array([tokens])

    # The vocabulary decode returns a NoteSequence if configured correctly
    return self.vocab.decode(tokens)

_mt3_model = None

def get_mt3_model():
    global _mt3_model
    if _mt3_model is None:
        print('Loading MT3 model (Restoring from checkpoint)...')
        _mt3_model = InferenceModel('/content/checkpoints/mt3', 'mt3')
        print('MT3 model ready.')
    return _mt3_model

def note_sequence_to_notes(ns):
    # [FIX] Handle cases where ns might be a list or a NoteSequence
    if isinstance(ns, list):
        return ns
    notes = []
    for n in ns.notes:
        if n.is_drum: continue
        notes.append({
            'onset': round(float(n.start_time), 4),
            'offset': round(float(n.end_time), 4),
            'pitch_midi': int(n.pitch),
        })
    return sorted(notes, key=lambda x: x['onset'])

def run_mt3(wav_path):
    import soundfile as sf
    model = get_mt3_model()
    audio, sr = sf.read(str(wav_path))
    if audio.ndim > 1: audio = audio.mean(axis=1)
    ns = model(audio)
    return note_sequence_to_notes(ns)

# Quick smoke test with full traceback capture
try:
    test_wav = PROCESSED_DIR / processed[0] / 'audio_16k.wav'
    test_notes = run_mt3(test_wav)
    print(f'Smoke test OK: {len(test_notes)} notes in "{processed[0]}"')
except Exception:
    print('MT3 Inference failed:')
    traceback.print_exc()

In [ ]:
# ── 9d: Run MT3 on all pieces ────────────────────────────────────────────────
import json
mt3_dir = PRED_DIR / 'mt3'
mt3_dir.mkdir(parents=True, exist_ok=True)

for entry in tqdm(dataset_info, desc='MT3'):
    stem = entry['stem']
    out  = mt3_dir / f'{stem}.json'
    if out.exists(): continue
    wav = PROCESSED_DIR / stem / 'audio_16k.wav'
    if not wav.exists(): continue
    try:
        notes = run_mt3(wav)
        with open(out, 'w') as f:
            json.dump(notes, f, indent=2)
    except Exception as e:
        print(f'  [ERROR] {stem}: {e}')

n_done = len(list(mt3_dir.glob('*.json')))
print(f'MT3 complete: {n_done}/{len(dataset_info)} predictions saved')


---
## 10 · Evaluate Baselines


| Block | Metrics | Tolerances |
|-------|---------|------------|
| Left | Precision / Recall / F-Measure / AOR | onset ±50ms, offset ±50ms or 20%, pitch ±0.5 st |
| Right | P / R / F (onset+pitch only) | onset ±50ms, pitch ±0.5 st, offset ignored |

In [ ]:
import mir_eval

def notes_to_arrays(notes):
    if not notes:
        return np.zeros((0, 2)), np.zeros(0)
    iv = np.array([[n['onset'], n['offset']] for n in notes], dtype=float)
    p  = np.array([float(n['pitch_midi']) for n in notes], dtype=float)
    return iv, p

def evaluate_piece(ref_notes, est_notes):
    ref_iv, ref_p = notes_to_arrays(ref_notes)
    est_iv, est_p = notes_to_arrays(est_notes)
    if len(ref_iv) == 0 and len(est_iv) == 0:
        return {'P': 1.0, 'R': 1.0, 'F': 1.0, 'AOR': 1.0,
                'P_op': 1.0, 'R_op': 1.0, 'F_op': 1.0}
    # onset + offset + pitch
    p, r, f, aor = mir_eval.transcription.precision_recall_f1_overlap(
        ref_iv, ref_p, est_iv, est_p,
        onset_tolerance=ONSET_TOL, pitch_tolerance=PITCH_TOL,
        offset_ratio=OFFSET_RATIO, offset_min_tolerance=OFFSET_MIN,
    )
    # onset + pitch only
    p_op, r_op, f_op, _ = mir_eval.transcription.precision_recall_f1_overlap(
        ref_iv, ref_p, est_iv, est_p,
        onset_tolerance=ONSET_TOL, pitch_tolerance=PITCH_TOL,
        offset_ratio=None,
    )
    return {'P': p, 'R': r, 'F': f, 'AOR': aor,
            'P_op': p_op, 'R_op': r_op, 'F_op': f_op,
            'n_ref': len(ref_notes), 'n_est': len(est_notes)}

def evaluate_model(pred_dir, model_name):
    results = []
    for entry in dataset_info:
        stem     = entry['stem']
        ref_path = PROCESSED_DIR / stem / 'notes.json'
        est_path = pred_dir / f'{stem}.json'
        if not ref_path.exists() or not est_path.exists(): continue
        with open(ref_path) as f: ref = json.load(f)
        with open(est_path) as f: est = json.load(f)
        m = evaluate_piece(ref, est)
        m['stem'] = stem
        results.append(m)
    keys = ['P', 'R', 'F', 'AOR', 'P_op', 'R_op', 'F_op']
    avg  = {k: float(np.mean([r[k] for r in results])) for k in keys}
    avg['n_pieces'] = len(results)
    return {'model': model_name, 'aggregate': avg, 'per_piece': results}

print('Evaluating pYIN ...')
pyin_results  = evaluate_model(pyin_dir,  'pYIN')
print('Evaluating CREPE Notes ...')
crepe_results = evaluate_model(crepe_dir, 'CREPE Notes')
print('Evaluating MT3 ...')
mt3_results   = evaluate_model(mt3_dir,   'MT3')

all_results = [pyin_results, crepe_results, mt3_results]
with open(EVAL_DIR / 'results.json', 'w') as f:
    json.dump(all_results, f, indent=2)

print()
hdr = f"  {'Model':<16} {'P':>6} {'R':>6} {'F1':>6} {'AOR':>6}   {'P(op)':>6} {'R(op)':>6} {'F1(op)':>7}"
print('=' * len(hdr))
print(hdr)
print(f"  {'':16} {'':6} {'':6} {'':6} {'':6}   {'<-- onset+pitch only':>21}")
print('=' * len(hdr))
for r in all_results:
    ag = r['aggregate']
    print(f"  {r['model']:<16} {ag['P']:>6.3f} {ag['R']:>6.3f} {ag['F']:>6.3f} "
          f"{ag['AOR']:>6.3f}   {ag['P_op']:>6.3f} {ag['R_op']:>6.3f} {ag['F_op']:>7.3f}")
print('=' * len(hdr))
print(f'  Left: onset+offset+pitch  |  Tolerances: onset +/-{ONSET_TOL*1000:.0f}ms, '
      f'offset +/-{OFFSET_MIN*1000:.0f}ms or {OFFSET_RATIO*100:.0f}%, pitch +/-{PITCH_TOL} st')

---
## 11 · Plot Results

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

COLORS = ['#4C72B0', '#DD8452', '#55A868']   # pYIN, CREPE Notes, MT3

# ── Grouped bar chart: all aggregate metrics ──────────────────────────────────
metrics       = ['P',         'R',      'F',           'AOR',               'F_op']
metric_labels = ['Precision', 'Recall', 'F-Measure',   'Avg Overlap\nRatio', 'F1\n(onset+pitch)']
n_models      = len(all_results)
x             = np.arange(len(metrics))
bar_w         = 0.7 / n_models

fig, ax = plt.subplots(figsize=(13, 5))
for i, (res, color) in enumerate(zip(all_results, COLORS)):
    offset = (i - n_models / 2 + 0.5) * bar_w
    vals   = [res['aggregate'][m] for m in metrics]
    bars   = ax.bar(x + offset, vals, bar_w, label=res['model'],
                    color=color, alpha=0.85, edgecolor='white', linewidth=0.5)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.012,
                f'{val:.3f}', ha='center', va='bottom', fontsize=7.5)

ax.set_xticks(x)
ax.set_xticklabels(metric_labels, fontsize=10)
ax.set_ylim(0, 1.18)
ax.set_ylabel('Score', fontsize=11)
ax.set_title('ITM-Flute-99 — Baseline Comparison  (macro avg, LOO-CV)',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10, loc='upper right')
ax.yaxis.set_major_formatter(mtick.FormatStrFormatter('%.2f'))
ax.grid(axis='y', alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
# Dashed divider between onset+offset block and onset-only block
ax.axvline(x=3.5, color='grey', linestyle='--', linewidth=0.8, alpha=0.5)
ax.text(3.55, 1.12, 'onset+pitch\nonly', fontsize=8, color='grey')
plt.tight_layout()
plt.savefig(EVAL_DIR / 'baseline_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Per-piece F1 line plot ─────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 4))
stems = [r['stem'] for r in all_results[0]['per_piece']]
x_pos = np.arange(len(stems))

for res, color in zip(all_results, COLORS):
    by_stem  = {r['stem']: r['F'] for r in res['per_piece']}
    f_scores = [by_stem.get(s, 0.0) for s in stems]
    ax.plot(x_pos, f_scores, 'o-', label=res['model'],
            color=color, alpha=0.85, markersize=4, linewidth=1.5)

ax.set_xlabel('Piece index', fontsize=11)
ax.set_ylabel('F-Measure', fontsize=11)
ax.set_title('Per-piece F-Measure — ITM-Flute-99', fontsize=12, fontweight='bold')
ax.set_xlim(-0.5, len(stems) - 0.5)
ax.set_ylim(0, 1.05)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(EVAL_DIR / 'per_piece_f1.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Figures saved to {EVAL_DIR}')

---
## 12 · Save All Outputs to Google Drive

In [ ]:
import shutil

DRIVE_OUT = Path('/content/drive/MyDrive/DL Project/crepe_yolo_outputs')
DRIVE_OUT.mkdir(parents=True, exist_ok=True)

shutil.copytree(SPLIT_DIR, DRIVE_OUT / 'configs',     dirs_exist_ok=True)
shutil.copytree(EVAL_DIR,  DRIVE_OUT / 'evaluation',  dirs_exist_ok=True)
shutil.copytree(PRED_DIR,  DRIVE_OUT / 'predictions', dirs_exist_ok=True)

# Copy cleaned notes.json only (skip large audio_16k.wav)
proc_out = DRIVE_OUT / 'processed'
for p in PROCESSED_DIR.rglob('notes.json'):
    dst = proc_out / p.relative_to(PROCESSED_DIR)
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(p, dst)

print(f'Saved to Google Drive: {DRIVE_OUT}')
print('Contents:')
for item in sorted(DRIVE_OUT.iterdir()):
    n_files = len(list(item.rglob('*'))) if item.is_dir() else 1
    print(f'  {item.name}/  ({n_files} files)')
print()
print('Share the "crepe_yolo_outputs" folder with your teammates:')
print('  configs/     -> train/test splits (for person 1 + person 2)')
print('  processed/   -> cleaned notes.json per piece (for person 1)')
print('  predictions/ -> baseline prediction JSONs (for evaluation comparison)')
print('  evaluation/  -> results.json + figures')
